# Section 03 lab — DataFrame transformations

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Select and rename columns

Begin transformation work by controlling the columns in the result.


### Step 1 — Create an employee DataFrame

The visible input makes each selected column easy to check.


In [ ]:
employees = spark.createDataFrame([(1, 'Asha', 'Engineering', 90000), (2, 'Bilal', 'Sales', 72000), (3, 'Chen', 'Finance', 81000)], ['employee_id', 'name', 'department', 'salary'])
employees.show()


### Step 2 — Select two columns

select removes columns that are not requested.


In [ ]:
names = employees.select('employee_id', 'name')
names.show()


### Step 3 — Rename an expression with alias

alias gives the output column a business-friendly name.


In [ ]:
from pyspark.sql import functions as F
salary_view = employees.select('name', F.col('salary').alias('annual_salary'))
salary_view.show()


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
employees.createOrReplaceTempView('section03_employees')
sql_result = spark.sql('SELECT employee_id, name, salary AS annual_salary FROM section03_employees ORDER BY employee_id')
sql_result.show()
assert sql_result.count() == salary_view.count()


## Exercise 2 — Filter rows

Keep only rows that satisfy a condition.


### Step 1 — Create order rows

The statuses and amounts support simple and compound conditions.


In [ ]:
orders = spark.createDataFrame([('O1', 'COMPLETED', 500.0), ('O2', 'CANCELLED', 700.0), ('O3', 'COMPLETED', 1200.0), ('O4', 'RETURNED', 300.0)], ['order_id', 'status', 'amount'])


### Step 2 — Apply one condition

This keeps completed orders only.


In [ ]:
from pyspark.sql import functions as F
completed = orders.filter(F.col('status') == 'COMPLETED')
completed.show()


### Step 3 — Combine two conditions

Parentheses make the AND logic explicit.


In [ ]:
large_completed = orders.filter((F.col('status') == 'COMPLETED') & (F.col('amount') >= 1000))
large_completed.show()
assert large_completed.first()['order_id'] == 'O3'


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
orders.createOrReplaceTempView('section03_orders')
sql_result = spark.sql("SELECT order_id, status, amount FROM section03_orders WHERE status = 'COMPLETED' AND amount >= 1000 ORDER BY order_id")
sql_result.show()
assert sql_result.count() == large_completed.count()


## Exercise 3 — Create columns with withColumn

Convert text to numbers before calculating a new value.


### Step 1 — Create text-based quantity and price fields

This resembles values arriving from CSV.


In [ ]:
lines = spark.createDataFrame([('L1', '2', '500.00'), ('L2', '3', '750.00')], ['line_id', 'quantity', 'unit_price'])
lines.printSchema()


### Step 2 — Convert the source fields

try_cast produces typed columns while keeping the source columns visible.


In [ ]:
from pyspark.sql import functions as F
typed = (lines.withColumn('qty', F.expr('try_cast(quantity as int)')).withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
typed.printSchema()


### Step 3 — Calculate line totals

The derived column now uses numeric inputs.


In [ ]:
calculated = typed.withColumn('line_total', F.col('qty') * F.col('price'))
calculated.select('line_id', 'qty', 'price', 'line_total').show()


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
lines.createOrReplaceTempView('section03_lines')
sql_result = spark.sql("""
SELECT line_id,
       TRY_CAST(quantity AS INT) AS qty,
       TRY_CAST(unit_price AS DECIMAL(12,2)) AS price,
       TRY_CAST(quantity AS INT) * TRY_CAST(unit_price AS DECIMAL(12,2)) AS line_total
FROM section03_lines
ORDER BY line_id
""")
sql_result.show()
assert sql_result.count() == calculated.count()


## Exercise 4 — Sort rows with orderBy

Control how results are presented after selecting the required columns.


### Step 1 — Create category revenue rows

Two categories share a revenue value so a secondary sort is visible.


In [ ]:
revenue = spark.createDataFrame([('Audio', 2500.0), ('Accessories', 1200.0), ('Electronics', 2500.0), ('Stationery', 300.0)], ['category', 'revenue'])


### Step 2 — Sort in descending order

Highest revenue appears first.


In [ ]:
from pyspark.sql import functions as F
revenue.orderBy(F.desc('revenue')).show()


### Step 3 — Add a secondary sort

Category resolves ties consistently.


In [ ]:
ranked = revenue.orderBy(F.desc('revenue'), F.asc('category'))
ranked.show()
assert ranked.first()['category'] == 'Audio'


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
revenue.createOrReplaceTempView('section03_revenue')
sql_result = spark.sql('SELECT category, revenue FROM section03_revenue ORDER BY revenue DESC, category ASC')
sql_result.show()
assert sql_result.first()['category'] == ranked.first()['category']


## Exercise 5 — Build a complete transformation chain

Combine only the operations introduced in this section.


### Step 1 — Read the transaction source with an explicit schema

The source path and fields are defined within this exercise.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([
    StructField('transaction_id', StringType()), StructField('transaction_date', StringType()),
    StructField('customer_id', StringType()), StructField('product_id', StringType()),
    StructField('quantity', StringType()), StructField('unit_price', StringType()),
    StructField('status', StringType())])
raw = spark.read.schema(schema).option('header', True).csv('../data/transactions.csv')


### Step 2 — Select and filter

Keep the fields needed for completed-order quantities.


In [ ]:
from pyspark.sql import functions as F
completed = raw.select('transaction_id', 'quantity', 'status').filter(F.col('status') == 'COMPLETED')


### Step 3 — Convert, filter and sort

The final result uses select, filter, withColumn and orderBy in one readable chain.


In [ ]:
result = (completed.withColumn('qty', F.expr('try_cast(quantity as int)')).filter(F.col('qty').isNotNull() & (F.col('qty') > 0)).orderBy(F.desc('qty'), F.asc('transaction_id')))
result.show(15)
assert result.filter(F.col('qty').isNull()).count() == 0


### Paired literal SQL solution

Register the DataFrame from this activity as a temporary view and reproduce the result with spark.sql.


In [ ]:
raw.createOrReplaceTempView('section03_transactions')
sql_result = spark.sql("""
SELECT transaction_id, quantity, status, TRY_CAST(quantity AS INT) AS qty
FROM section03_transactions
WHERE status = 'COMPLETED'
  AND TRY_CAST(quantity AS INT) IS NOT NULL
  AND TRY_CAST(quantity AS INT) > 0
ORDER BY qty DESC, transaction_id ASC
""")
sql_result.show(15)
assert sql_result.count() == result.count()
